# 07 — BioASQ + PubMed + MedQuAD + HealthCareMagic Retrieval Dataset

Builds `data/processed/bioasq_pubmed_retrieval.csv` — a retrieval-ready dataset that consolidates
four medical QA sources into a single file. Each row pairs a medical question with:
- Its **ground-truth answer** (exact + ideal)
- The **supporting PubMed snippets/abstracts** used as retrieval context (BioASQ rows only)
- A structured **Plan of Action (POA)** column derived from the ideal answer

### Sources
| Source | Role |
|---|---|
| `data/raw/BioASQ/*_golden.json` | Gold-standard test questions with rich metadata (snippets + PMIDs) |
| `data/raw/BioASQ/BioASQ-train-factoid-{4-7}b.json` | Factoid training questions (SQuAD-style) |
| `data/raw/BioASQ/BioASQ-train-yesno-7b.json` | Yes/No training questions (SQuAD-style) |
| `data/raw/BioASQ/BioASQ-test-factoid-*.json` | Factoid test questions — questions only, no answers |
| `data/raw/BioASQ/BioASQ-test-yesno-7b.json` | Yes/No test questions — questions only, no answers |
| `data/raw/pubmed_abstracts.csv` | PubMed abstract corpus (8 topics, wide format) — joined on PMID |
| `data/raw/medquad.csv` | NIH MedQuAD QA pairs (GHR, GARD, NIDDK, NINDS, etc.) |
| `data/raw/meddialog/HealthCareMagic-100k.json` | Real doctor–patient Q&A (HealthCareMagic.com) |

### Output schema
| Column | Description |
|---|---|
| `question_id` | Unique question ID (BioASQ ID / `mq_<hash>` / `hcm_<hash>`) |
| `question` | Question text |
| `answer_type` | `factoid` / `list` / `yesno` / `summary` / `open_qa` |
| `ground_truth_answer` | Exact answer (string) |
| `ideal_answer` | Natural-language ideal answer |
| `snippets_text` | `\|\|\|`-separated supporting passages (BioASQ only) |
| `snippet_pmids` | Pipe-separated PMIDs for those passages |
| `all_doc_pmids` | All document PMIDs referenced |
| `pubmed_title` | Matched PubMed title (if available) |
| `pubmed_abstract` | Matched PubMed abstract (if available) |
| `pubmed_topic` | Topic category from pubmed_abstracts.csv |
| `plan_of_action` | Structured POA derived from ideal answer |
| `source_file` | Which source file this row came from |

In [ ]:
import json
import ast
import re
import hashlib
import pandas as pd
from pathlib import Path

ROOT           = Path("..")
RAW_BIOASQ     = ROOT / "data" / "raw" / "BioASQ"
RAW_PUBMED     = ROOT / "data" / "raw" / "pubmed_abstracts.csv"
RAW_MEDQUAD    = ROOT / "data" / "raw" / "medquad.csv"
RAW_HCM        = ROOT / "data" / "raw" / "meddialog" / "HealthCareMagic-100k.json"
OUT_DIR        = ROOT / "data" / "processed"
OUT_PATH       = OUT_DIR / "bioasq_pubmed_retrieval.csv"

print("BioASQ dir  :", RAW_BIOASQ.resolve())
print("PubMed CSV  :", RAW_PUBMED.resolve())
print("MedQuAD CSV :", RAW_MEDQUAD.resolve())
print("HCM JSON    :", RAW_HCM.resolve())
print("Output      :", OUT_PATH.resolve())

## 1. Helper utilities

In [ ]:
def pmid_from_url(url: str) -> str:
    """Extract numeric PMID from a PubMed URL or return the raw string."""
    if not isinstance(url, str):
        return ""
    m = re.search(r"pubmed/(\d+)", url)
    return m.group(1) if m else url.strip()


def flatten_exact_answer(exact_answer, answer_type: str) -> str:
    """
    Normalise exact_answer across BioASQ question types:
      factoid / list  → list-of-lists  e.g. [["ans1","syn"], ["ans2"]]
      yesno           → plain string   "yes" / "no"
      summary         → absent / None
    Returns a readable string.
    """
    if exact_answer is None:
        return ""
    if isinstance(exact_answer, str):
        return exact_answer.strip()
    if isinstance(exact_answer, list):
        # Flatten inner lists with " / " and join outer with " | "
        parts = []
        for item in exact_answer:
            if isinstance(item, list):
                parts.append(" / ".join(str(x) for x in item))
            else:
                parts.append(str(item))
        return " | ".join(parts)
    return str(exact_answer)


def first_ideal_answer(ideal_answer) -> str:
    """Return the first ideal answer string."""
    if not ideal_answer:
        return ""
    if isinstance(ideal_answer, list):
        return ideal_answer[0].strip()
    return str(ideal_answer).strip()


def short_hash(text: str, length: int = 12) -> str:
    """Return a short MD5 hex digest of `text`."""
    return hashlib.md5(text.encode("utf-8", errors="replace")).hexdigest()[:length]


def derive_plan_of_action(ideal_answer: str, answer_type: str, exact_answer: str) -> str:
    """
    Build a structured Plan-of-Action (POA) string from the available answer fields.

    POA template:
      [Step 1] Identify the clinical/research question.
      [Step 2] Retrieve supporting evidence from PubMed snippets.
      [Step 3] Ground-truth answer: <exact_answer>
      [Step 4] Synthesise: <condensed ideal_answer>
      [Step 5] Recommend: <action / decision based on answer type>
    """
    if not ideal_answer and not exact_answer:
        return ""

    # Condense ideal answer to ≤200 chars for POA
    summary = ideal_answer[:200].rstrip() + ("..." if len(ideal_answer) > 200 else "")

    if answer_type == "yesno":
        recommendation = (
            f"Clinical recommendation: The answer is '{exact_answer.upper()}'. "
            "Proceed or withhold intervention accordingly."
        )
    elif answer_type == "factoid":
        recommendation = (
            f"Clinical recommendation: The specific entity/value is '{exact_answer}'. "
            "Use this as the target in downstream queries or treatment decisions."
        )
    elif answer_type == "list":
        items = exact_answer.split(" | ") if exact_answer else []
        recommendation = (
            f"Clinical recommendation: Consider each of the following ({len(items)} items): "
            + "; ".join(items[:5])
            + (". (truncated)" if len(items) > 5 else ".")
        )
    elif answer_type == "open_qa":  # FIX: was missing, fell through to summary branch
        recommendation = (
            "Clinical recommendation: Review the full answer above and follow the "
            "treating clinician's guidance before taking any action."
        )
    else:  # summary (and any unknown type)
        recommendation = (
            "Clinical recommendation: Review the synthesised evidence above "
            "and formulate a management plan with the treating clinician."
        )

    poa = (
        "[Step 1] Identify the clinical/research question.\n"
        "[Step 2] Retrieve supporting evidence from PubMed snippets.\n"
        f"[Step 3] Ground-truth answer: {exact_answer}\n"
        f"[Step 4] Synthesise: {summary}\n"
        f"[Step 5] {recommendation}"
    )
    return poa


print("Helpers defined.")

## 2. Parse BioASQ *golden* files (native format)

These are the richest files: `7B_golden.json`, `6B{1-5}_golden.json`, etc.  
Each question carries: `body`, `type`, `documents`, `snippets`, `ideal_answer`, `exact_answer`.

In [ ]:
def parse_golden_file(path: Path) -> list[dict]:
    """
    Parse a native BioASQ golden JSON file.
    Returns a list of flat record dicts.
    """
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    records = []
    questions = data.get("questions", [])

    for q in questions:
        qid          = q.get("id", "")
        body         = q.get("body", "")
        qtype        = q.get("type", "unknown")
        documents    = q.get("documents", [])
        snippets     = q.get("snippets", [])
        ideal_answer = first_ideal_answer(q.get("ideal_answer", ""))
        exact        = flatten_exact_answer(q.get("exact_answer", None), qtype)

        # Flatten snippets
        snippet_texts = [s.get("text", "").strip() for s in snippets]
        snippet_pmids = [pmid_from_url(s.get("document", "")) for s in snippets]

        # All referenced PMIDs
        all_pmids = [pmid_from_url(d) for d in documents]

        records.append({
            "question_id"        : qid,
            "question"           : body,
            "answer_type"        : qtype,
            "ground_truth_answer": exact,
            "ideal_answer"       : ideal_answer,
            "snippets_text"      : " ||| ".join(snippet_texts),
            "snippet_pmids"      : "|".join(snippet_pmids),
            "all_doc_pmids"      : "|".join(all_pmids),
            "source_file"        : path.name,
        })

    return records


# Collect all golden files
golden_files = sorted(RAW_BIOASQ.glob("*_golden.json"))
print(f"Found {len(golden_files)} golden files:")
for f in golden_files:
    print(" ", f.name)

In [ ]:
all_golden_records = []
for gf in golden_files:
    recs = parse_golden_file(gf)
    print(f"  {gf.name:45s} → {len(recs):4d} questions")
    all_golden_records.extend(recs)

df_golden = pd.DataFrame(all_golden_records)
print(f"\nTotal golden records : {len(df_golden):,}")
print(df_golden["answer_type"].value_counts())

## 3. Parse BioASQ *train* files (SQuAD-style)

These files use a different schema — SQuAD-style with `data → paragraphs → context / qas`.  
Multiple paragraphs exist per question (one per supporting passage).  
We de-duplicate by `question_id` and aggregate all contexts.

In [ ]:
def parse_squad_file(path: Path, answer_type: str) -> list[dict]:
    """
    Parse a SQuAD-style BioASQ train or test JSON file.
    `answer_type` is 'factoid' or 'yesno' (inferred from filename).
    Test files have no `answers` field — ground_truth_answer will be empty.
    Returns one record per unique question (contexts aggregated).
    """
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    # Accumulate per question_id
    q_map: dict[str, dict] = {}  # base_id → record

    for group in data.get("data", []):
        for para in group.get("paragraphs", []):
            context = para.get("context", "").strip()
            for qa in para.get("qas", []):
                full_id  = qa.get("id", "")
                # Strip the trailing _NNN paragraph index to get the base question id
                base_id  = re.sub(r"_\d{3}$", "", full_id)
                question = qa.get("question", "").strip()

                # Extract answer depending on type — test files have no answers key
                raw_answers = qa.get("answers", None)
                if raw_answers is None:
                    ans_text = ""  # test-set question: no ground truth available
                elif answer_type == "yesno":
                    # train yesno: answers is a plain string "yes" / "no"
                    ans_text = raw_answers if isinstance(raw_answers, str) else ""
                else:
                    # train factoid: answers is a list of {text, answer_start}
                    if isinstance(raw_answers, list) and raw_answers:
                        ans_text = raw_answers[0].get("text", "")
                    else:
                        ans_text = ""

                if base_id not in q_map:
                    q_map[base_id] = {
                        "question_id"        : base_id,
                        "question"           : question,
                        "answer_type"        : answer_type,
                        "ground_truth_answer": ans_text,
                        "ideal_answer"       : ans_text,   # no long answer in SQuAD files
                        "snippets_text"      : context,
                        "snippet_pmids"      : "",         # not available in SQuAD format
                        "all_doc_pmids"      : "",
                        "source_file"        : path.name,
                        "_contexts"          : [context],
                    }
                else:
                    # Append additional context paragraphs
                    q_map[base_id]["_contexts"].append(context)

    # Finalise: join all contexts
    records = []
    for rec in q_map.values():
        all_ctx = list(dict.fromkeys(rec.pop("_contexts")))  # deduplicate preserving order
        rec["snippets_text"] = " ||| ".join(all_ctx)
        records.append(rec)

    return records


# Discover all train files (glob matches 4b, 5b, 6b, 7b)
factoid_train_files = sorted(RAW_BIOASQ.glob("BioASQ-train-factoid-*.json"))
yesno_train_files   = sorted(RAW_BIOASQ.glob("BioASQ-train-yesno-*.json"))
print("Factoid train files:", [f.name for f in factoid_train_files])
print("Yesno  train files :", [f.name for f in yesno_train_files])

In [ ]:
all_train_records = []

for tf in factoid_train_files:
    recs = parse_squad_file(tf, answer_type="factoid")
    print(f"  {tf.name:45s} → {len(recs):4d} questions")
    all_train_records.extend(recs)

for tf in yesno_train_files:
    recs = parse_squad_file(tf, answer_type="yesno")
    print(f"  {tf.name:45s} → {len(recs):4d} questions")
    all_train_records.extend(recs)

df_train = pd.DataFrame(all_train_records)
print(f"\nTotal train records  : {len(df_train):,}")
print(df_train["answer_type"].value_counts())

## 4. Parse BioASQ *test* files (SQuAD-style, questions only)

Test files follow the same SQuAD schema but have **no `answers` field** — they are question-only.  
`ground_truth_answer` and `ideal_answer` will be empty strings for these rows.

In [ ]:
# Discover all test files
factoid_test_files = sorted(RAW_BIOASQ.glob("BioASQ-test-factoid-*.json"))
yesno_test_files   = sorted(RAW_BIOASQ.glob("BioASQ-test-yesno-*.json"))
print("Factoid test files:", [f.name for f in factoid_test_files])
print("Yesno  test files :", [f.name for f in yesno_test_files])

all_test_records = []

for tf in factoid_test_files:
    recs = parse_squad_file(tf, answer_type="factoid")
    print(f"  {tf.name:45s} → {len(recs):4d} questions")
    all_test_records.extend(recs)

for tf in yesno_test_files:
    recs = parse_squad_file(tf, answer_type="yesno")
    print(f"  {tf.name:45s} → {len(recs):4d} questions")
    all_test_records.extend(recs)

df_test = pd.DataFrame(all_test_records) if all_test_records else pd.DataFrame(columns=list(df_train.columns))
print(f"\nTotal test records   : {len(df_test):,}")
if len(df_test):
    print(df_test["answer_type"].value_counts())

## 5. Merge BioASQ golden + train + test, deduplicate

In [ ]:
df_bioasq = pd.concat([df_golden, df_train, df_test], ignore_index=True)

print(f"Combined (before dedup): {len(df_bioasq):,}")

# Drop duplicates on question_id — golden > train > test (golden appears first)
df_bioasq = df_bioasq.drop_duplicates(subset="question_id", keep="first")

print(f"After dedup by question_id: {len(df_bioasq):,}")
print("Answer type breakdown:")
print(df_bioasq["answer_type"].value_counts())

## 6. Parse PubMed abstracts CSV — wide → long

The CSV is wide-format with 8 topic columns. Each cell is a Python tuple string `([abstract], 'title')`.  
We melt it to one row per document and map PMIDs from the corresponding URL columns.

In [ ]:
TOPIC_COLS = [
    "deep_learning", "covid_19", "human_connectome", "virtual_reality",
    "brain_machine_interfaces", "electroactive_polymers", "pedot_electrodes",
    "neuroprosthetics",
]
LINK_COLS = [f"{t}_links" for t in TOPIC_COLS]


def safe_literal_eval(val):
    """
    Parse the cell value as a Python literal.
    Returns (abstract_str, title_str) — both guaranteed to be strings.

    FIX: the original code returned ([], "") on failure (a list, not a string),
    causing  AttributeError: 'list' object has no attribute 'strip'  in the
    caller.  Now always returns a 2-tuple of strings.
    """
    if not isinstance(val, str) or not val.strip():
        return ("", "")  # FIX: was ([], "") — list caused .strip() to crash
    try:
        result = ast.literal_eval(val.strip())
        # Expected: ([abstract_text, ...], title_string)
        if isinstance(result, tuple) and len(result) == 2:
            abstracts, title = result
            abstract = abstracts[0] if isinstance(abstracts, list) and abstracts else ""
            return (str(abstract), str(title))
    except Exception:
        pass
    return ("", str(val)[:120])  # FIX: was ("", str(val)[:120]) — already a string, OK


df_pub_raw = pd.read_csv(RAW_PUBMED)
print(f"PubMed CSV shape: {df_pub_raw.shape}")
print("Columns:", list(df_pub_raw.columns))

In [ ]:
pubmed_records = []

for _, row in df_pub_raw.iterrows():
    for topic, link_col in zip(TOPIC_COLS, LINK_COLS):
        cell     = row.get(topic, "")
        url      = str(row.get(link_col, "")).strip()
        pmid     = pmid_from_url(url)

        abstract, title = safe_literal_eval(cell)  # always returns (str, str) now

        if not pmid and not abstract and not title:
            continue  # skip fully empty cells

        pubmed_records.append({
            "pmid"           : pmid,
            "pubmed_title"   : title.strip(),
            "pubmed_abstract": abstract.strip(),
            "pubmed_topic"   : topic,
            "pubmed_url"     : url,
        })

df_pubmed = pd.DataFrame(pubmed_records)

# Deduplicate: keep one row per PMID (first occurrence wins)
df_pubmed = df_pubmed[df_pubmed["pmid"].str.len() > 0]
df_pubmed_dedup = df_pubmed.drop_duplicates(subset="pmid", keep="first")

print(f"PubMed long-format rows  : {len(df_pubmed):,}")
print(f"After PMID dedup         : {len(df_pubmed_dedup):,}")
print("Topics:", df_pubmed["pubmed_topic"].value_counts().to_dict())

## 7. Parse MedQuAD

`medquad.csv` has columns: `question`, `answer`, `source`, `focus_area`.  
We assign `answer_type = open_qa` and generate a short MD5-based `question_id` prefixed `mq_`.

In [ ]:
df_mq_raw = pd.read_csv(RAW_MEDQUAD, dtype=str).fillna("")
print(f"MedQuAD raw shape: {df_mq_raw.shape}")
print("Sources:", df_mq_raw["source"].value_counts().to_dict())

mq_records = []
for _, row in df_mq_raw.iterrows():
    question = row["question"].strip()
    answer   = row["answer"].strip()
    source   = row["source"].strip()

    if not question:
        continue

    qid = "mq_" + short_hash(question)
    source_label = f"medquad.csv [{source}]" if source else "medquad.csv"

    mq_records.append({
        "question_id"        : qid,
        "question"           : question,
        "answer_type"        : "open_qa",
        "ground_truth_answer": answer,
        "ideal_answer"       : answer,
        "snippets_text"      : "",
        "snippet_pmids"      : "",
        "all_doc_pmids"      : "",
        "source_file"        : source_label,
    })

df_medquad = pd.DataFrame(mq_records)
print(f"\nMedQuAD records  : {len(df_medquad):,}")
print(df_medquad["source_file"].value_counts())

## 8. Parse HealthCareMagic

`HealthCareMagic-100k.json` is a list of dicts with keys `instruction`, `input`, `output`.  
- `input`  → `question`  
- `output` → `ground_truth_answer` and `ideal_answer`  
`question_id` is prefixed `hcm_` with a short MD5 hash of the input text.

In [ ]:
with open(RAW_HCM, encoding="utf-8") as f:
    hcm_raw = json.load(f)

print(f"HealthCareMagic raw records: {len(hcm_raw):,}")
print("Sample keys:", list(hcm_raw[0].keys()))

hcm_records = []
for entry in hcm_raw:
    question = entry.get("input", "").strip()
    answer   = entry.get("output", "").strip()

    if not question:
        continue

    qid = "hcm_" + short_hash(question)

    hcm_records.append({
        "question_id"        : qid,
        "question"           : question,
        "answer_type"        : "open_qa",
        "ground_truth_answer": answer,
        "ideal_answer"       : answer,
        "snippets_text"      : "",
        "snippet_pmids"      : "",
        "all_doc_pmids"      : "",
        "source_file"        : RAW_HCM.name,
    })

df_hcm = pd.DataFrame(hcm_records)
print(f"\nHealthCareMagic records: {len(df_hcm):,}")

## 9. Join BioASQ → PubMed abstracts via PMID

Strategy:
1. Extract the first available PMID from `snippet_pmids`; fall back to `all_doc_pmids`.
2. Left-join to the deduplicated PubMed table on PMID.
3. For questions with multiple PMIDs, keep the first successful match.

Note: `pubmed_abstracts.csv` covers 8 niche research topics so PMID overlap with BioASQ clinical questions is sparse.

In [ ]:
def first_pmid(pmid_string: str) -> str:
    """Return the first PMID from a pipe-separated string."""
    if not isinstance(pmid_string, str):
        return ""
    parts = [p.strip() for p in pmid_string.split("|") if p.strip()]
    return parts[0] if parts else ""


# Extract first snippet PMID; fall back to first doc PMID
df_bioasq["_join_pmid"] = df_bioasq["snippet_pmids"].apply(first_pmid)
mask_empty = df_bioasq["_join_pmid"] == ""
df_bioasq.loc[mask_empty, "_join_pmid"] = df_bioasq.loc[mask_empty, "all_doc_pmids"].apply(first_pmid)

print(f"BioASQ questions with a join PMID : {(df_bioasq['_join_pmid'] != '').sum():,}")
print(f"BioASQ questions without any PMID : {(df_bioasq['_join_pmid'] == '').sum():,}")

In [ ]:
# Merge BioASQ ← PubMed
df_bioasq_merged = df_bioasq.merge(
    df_pubmed_dedup[["pmid", "pubmed_title", "pubmed_abstract", "pubmed_topic"]],
    left_on="_join_pmid",
    right_on="pmid",
    how="left",
).drop(columns=["_join_pmid", "pmid"])

for col in ["pubmed_title", "pubmed_abstract", "pubmed_topic"]:
    df_bioasq_merged[col] = df_bioasq_merged[col].fillna("")

matched = (df_bioasq_merged["pubmed_abstract"] != "").sum()
print(f"BioASQ rows with matched PubMed abstract : {matched:,} / {len(df_bioasq_merged):,} ({matched/len(df_bioasq_merged)*100:.1f}%)")

# MedQuAD and HCM have no PMIDs — add empty PubMed columns so concat aligns
for df in [df_medquad, df_hcm]:
    df["pubmed_title"]    = ""
    df["pubmed_abstract"] = ""
    df["pubmed_topic"]    = ""

## 10. Combine all sources

In [ ]:
df_all = pd.concat([df_bioasq_merged, df_medquad, df_hcm], ignore_index=True)

print(f"Combined rows (all sources): {len(df_all):,}")
print("\nBy answer type:")
print(df_all["answer_type"].value_counts().to_string())
print("\nBy source file (top 15):")
print(df_all["source_file"].value_counts().head(15).to_string())

## 11. Generate Plan of Action (POA) column

In [ ]:
df_all["plan_of_action"] = df_all.apply(
    lambda row: derive_plan_of_action(
        ideal_answer=str(row["ideal_answer"]) if pd.notna(row["ideal_answer"]) else "",
        answer_type=str(row["answer_type"]),
        exact_answer=str(row["ground_truth_answer"]) if pd.notna(row["ground_truth_answer"]) else "",
    ),
    axis=1,
)

print(f"POA generated for {(df_all['plan_of_action'] != '').sum():,} rows.")
print("\nSample POA (first BioASQ row with POA):")
sample_poa = df_all[df_all["plan_of_action"] != ""].iloc[0]["plan_of_action"]
print(sample_poa)

## 12. Final column selection & quality checks

In [ ]:
FINAL_COLS = [
    "question_id",
    "question",
    "answer_type",
    "ground_truth_answer",
    "ideal_answer",
    "snippets_text",
    "snippet_pmids",
    "all_doc_pmids",
    "pubmed_title",
    "pubmed_abstract",
    "pubmed_topic",
    "plan_of_action",
    "source_file",
]

df_final = df_all[FINAL_COLS].copy()

# Drop rows with no question text (malformed entries)
before = len(df_final)
df_final = df_final[df_final["question"].fillna("").str.strip() != ""]
print(f"Removed {before - len(df_final)} empty-question rows.")

# Reset index
df_final = df_final.reset_index(drop=True)

print(f"\nFinal dataset shape: {df_final.shape}")
print("\nColumn null counts:")
print(df_final.isnull().sum().to_string())

In [ ]:
# Summary statistics
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Total questions          : {len(df_final):,}")
print(f"Unique question IDs      : {df_final['question_id'].nunique():,}")
print()
print("By answer type:")
print(df_final["answer_type"].value_counts().to_string())
print()
print("By source file (top 10):")
print(df_final["source_file"].value_counts().head(10).to_string())
print()
print(f"With ground_truth_answer : {(df_final['ground_truth_answer'].fillna('') != '').sum():,}")
print(f"With ideal_answer        : {(df_final['ideal_answer'].fillna('') != '').sum():,}")
print(f"With snippets            : {(df_final['snippets_text'].fillna('') != '').sum():,}")
print(f"With PubMed abstract     : {(df_final['pubmed_abstract'].fillna('') != '').sum():,}")
print(f"With POA                 : {(df_final['plan_of_action'].fillna('') != '').sum():,}")

## 13. Save dataset

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
df_final.to_csv(OUT_PATH, index=False, encoding="utf-8")
size_mb = OUT_PATH.stat().st_size / 1024 / 1024
print(f"Saved → {OUT_PATH}")
print(f"File size: {size_mb:.2f} MB")
print(f"Rows: {len(df_final):,}  |  Columns: {len(df_final.columns)}")

## 14. Inspect a few sample rows

In [ ]:
pd.set_option("display.max_colwidth", 120)

# FIX: display() is only available in Jupyter; fall back to print() otherwise
try:
    _display = display  # noqa: F821  (defined by IPython kernel)
except NameError:
    _display = print

print("── Factoid sample ──")
factoid_sample = df_final[df_final["answer_type"] == "factoid"].head(2)
_display(factoid_sample[["question", "ground_truth_answer", "ideal_answer", "pubmed_topic"]].T)

print("\n── Yes/No sample ──")
yesno_sample = df_final[df_final["answer_type"] == "yesno"].head(2)
_display(yesno_sample[["question", "ground_truth_answer", "ideal_answer"]].T)

print("\n── List sample ──")
list_sample = df_final[df_final["answer_type"] == "list"].head(2)
_display(list_sample[["question", "ground_truth_answer"]].T)

print("\n── Open QA sample (HealthCareMagic) ──")
openqa_sample = df_final[df_final["answer_type"] == "open_qa"].head(2)
_display(openqa_sample[["question", "ground_truth_answer", "source_file"]].T)

In [ ]:
# Show a complete record with POA
print("── Complete record (with POA) ──")
sample = df_final[df_final["plan_of_action"].fillna("") != ""].iloc[0]
for col in FINAL_COLS:
    val = sample[col]
    # Truncate long fields for display
    if isinstance(val, str) and len(val) > 300:
        val = val[:300] + "..."
    print(f"\n[{col}]\n{val}")

---
## Dataset complete

`data/processed/bioasq_pubmed_retrieval.csv` is ready for:
- **Ground-truth retrieval evaluation** — use `snippets_text` / `pubmed_abstract` as the retrieved context and `ground_truth_answer` / `ideal_answer` as the reference.
- **Plan-of-Action recommendation** — the `plan_of_action` column provides a structured 5-step POA for each question.
- **RAG pipeline input** — index `pubmed_abstract` + `snippets_text` in a vector store; use `question` as the query; compare retrieved answers against `ground_truth_answer`.
- **Fine-tuning** — `question` → `ideal_answer` pairs across all sources; filter by `answer_type` or `source_file` for task-specific subsets.